# Retrieval Evaluation

Evaluate fuzzy and semantic retrieval for the TM store.

In [89]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), "..", "src"))
from retrieve import load_translation_memory, fuzzy_retrieval, semantic_retrieval
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from qdrant_client.http import models
import uuid
import json
import pandas as pd

In [9]:
# Importing the translation memory
tm = load_translation_memory("../data/tm/translation_memory.jsonl")

In [ ]:
# Initializing model and client
model = SentenceTransformer("all-MiniLM-L6-v2")
client = QdrantClient(url="http://localhost:6333")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4581.27it/s]


In [30]:
# Converting gold_set to pandas df

gold_set = pd.read_json("../eval/gold_set.jsonl", lines=True)
gold_set.head()

,case_id,category,query,reference,edit_type,requires_agreement,flag,comment,previous_source,target_approved,source_file,base_id,source,target,acceptable_ids
0,f01,fuzzy_real,Pirate Flagship,Okręt flagowy piratów,reworded,no,ok,NaN,Fire Wraith,Ogniste widmo,pl_units.po,NaN,NaN,NaN,NaN
1,f02,fuzzy_real,plague ($type),zaraza ($type),added_clause,no,ok,NaN,plague,zaraza,pl_help.po,NaN,NaN,NaN,NaN
2,f03,fuzzy_real,Campaigns and Scenarios,Kampanie i scenariusze,placeholder,no,target_error,"It seems that ""Scenariusze"" shouldn't be capit...",<header>text='Campaigns and Scenarios'</header>,<header>text='Kampanie i Scenariusze'</header>,pl_help.po,NaN,NaN,NaN,NaN
3,f04,fuzzy_real,<ref>dst='..calendar' text='Calendar'</ref>\n<...,<ref>dst='..calendar' text='Kalendarz'</ref>\n...,added_clause,no,ok,NaN,<ref>dst='..geography' text='Geography'</ref>,<ref>dst='..geography' text='Geografia'</ref>,pl_help.po,NaN,NaN,NaN,NaN
4,f05,fuzzy_real,Time of Day Schedule,Grafik pór dnia,added_clause,yes,ok,NaN,Time of Day,Pora dnia,pl_help.po,NaN,NaN,NaN,NaN


In [34]:
# Filtering for edited category
gold_set_edited = gold_set[gold_set['category'] == 'edited']
gold_set_edited.head()


,case_id,category,query,reference,edit_type,requires_agreement,flag,comment,previous_source,target_approved,source_file,base_id,source,target,acceptable_ids
15,e01,edited,Water Drake,Smok wody,term,yes,ok,NaN,NaN,NaN,pl_units.po,ad6556ad9d1a7c5490aeee99cd5ae37f,Sky Drake,Smok przestworzy,[ad6556ad9d1a7c5490aeee99cd5ae37f]
16,e02,edited,This option directly connects you to the offic...,Ta opcja umożliwia połączenie się z oficjalnym...,reworded,yes,ok,Difficult agreement problem,NaN,NaN,pl_manual.po,f94cea635a476d688c00d55547dc991d,This option directly connects you to the offic...,Ta opcja umożliwia połączenie się z oficjalnym...,[f94cea635a476d688c00d55547dc991d]
17,e03,edited,Royal Architect,Królewski architekt,term,yes,ok,NaN,NaN,NaN,pl_units.po,be44cca919aed8f7ff929d174f33dbc5,Royal Warrior,Królewski wojownik,[be44cca919aed8f7ff929d174f33dbc5]
18,e04,edited,Dune Raider,Wydmowy jeździec,term,yes,ok,NaN,NaN,NaN,pl_units.po,582cac393fe26ef975ab0f963a5b6904,Dune Explorer,Wydmowy odkrywca,[582cac393fe26ef975ab0f963a5b6904]
19,e05,edited,Second Watch — fourth hour,Druga straż – czwarta godzina,reworded,no,ok,NaN,NaN,NaN,pl_help.po,6072f0716794dbd6ee7a93bef92b2e73,First Watch — Fourth Hour,Pierwsza straż — czwarta godzina,[6072f0716794dbd6ee7a93bef92b2e73]


In [36]:
# Counting the number of edited records
edited_count = gold_set_edited['query'].count()
print(f"Number of edited records: {edited_count}")

Number of edited records: 15


## Helper functions for metrics calculations

In [42]:
# Id ranking for fuzzy matches
test_query = gold_set_edited.iloc[0]['query']
print(test_query)

Water Drake


In [54]:
test_retrieval = fuzzy_retrieval(tm, test_query, 1)
print(test_retrieval[0][0])

66.66666666666667


In [97]:
def ranked_ids_fuzzy(query, k) -> list[str]:
    """Function returning top k id from fuzzy retrieval"""
    result = fuzzy_retrieval(tm, query, k)
    records = []
    for score, record in result:
        records.append(record['id'])
        
    return records

ranked_id_test = ranked_ids_fuzzy(test_query, 7)
print(ranked_id_test)

['6cc6bb3c77c7d12c68c0f9ad48ca457c', 'b3f504f9f5b11d7adfcf5fa81e2a7964', '58ecb10d84b4884e1657a76a7b2c152f', '5daef117699a4902980a91cf2216b740', 'dd99398ff131f3b1ecb26a578074b525', 'b8e903d5dd4c486bf3b2c7a418fc6841', '4d28cd285b4ee125e32741cfa188c0f9']


In [100]:
# Id ranking for semantic retrieval

def ranked_ids_semantic(query, k) -> list[str]:
    """Function returning top k id from semantic retrieval"""
    tm_index = {r["id"]: r for r in tm}
    result = semantic_retrieval(query, tm_index, model, client, k)
    records = []
    for score, record in result:
        records.append(record['id'])
            
    return records

ranked_id_semantic_test = ranked_ids_semantic(test_query, 7)
print(ranked_id_semantic_test)

['4d28cd285b4ee125e32741cfa188c0f9', 'a56c33315fdac1fb59947d3bd8689aaa', 'b3f504f9f5b11d7adfcf5fa81e2a7964', '649d435a01b5556d3cb137ceff202c24', 'ad6556ad9d1a7c5490aeee99cd5ae37f', 'c01826dddbd7ff74d6b3c4a0b847f380', '5daef117699a4902980a91cf2216b740']


In [101]:
# Does ranked_id exist in acceptable_id list?
for i in ranked_id_semantic_test:
    if i in gold_set_edited.iloc[0]['acceptable_ids']:
        print(True)
    else:
        print(False)



False
False
False
False
True
False
False


In [ ]:
def hit(ranked_ids, acceptable_ids, k) -> bool:
    """Returns true if any ranked_id exists in acceptable_ids - correct retrieval hit"""
    return any(s in acceptable_ids for s in ranked_ids[:k])

In [115]:
acceptable_ids_test = gold_set_edited.iloc[0]['acceptable_ids']
print(hit(ranked_id_semantic_test, acceptable_ids_test, 7))

True
